<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Why a config module?

MAX doesn't invent hyperparameters. It reads a HuggingFace-style `config.json` and asks *our* class to turn that into something the graph can use.

Two jobs:

1. **`SmiTedHFConfig`** — a plain dataclass mirroring the Hub fields we care about.
2. **`SmiTedModelConfig`** — the MAX `ArchConfig`: dtype, device, max sequence length, plus that HF subset.

If you only remember one number from this notebook: **`max_len = 202`**. Every batch is padded to that length. The autoencoder later flattens `202 × 768` into one vector.

## HuggingFace subset

We ignore most of `config.json`. The encode path only needs layer count, heads, embedding size, vocab, dropout (unused at inference), and the Performer feature count.

In [0]:
#| echo: false
#| output: asis
show_doc(SmiTedHFConfig)

---

### SmiTedHFConfig

```python
def SmiTedHFConfig(
    model_type:str='SMI-TED', architectures:list[str] | None=None, n_layer:int=12, n_head:int=12, n_embd:int=768,
    max_len:int=202, num_feats:int=32, d_dropout:float=0.1, vocab_size:int=2393
)->None:
```

*Subset of Hub ``config.json`` fields used by the MAX graph.*

Quick sanity check — defaults should look like a 12-layer BERT-sized encoder with a SMILES vocab:

## MAX model config

`initialize` is what the pipeline calls. It insists on:

- a quantization encoding (we use float32 / bfloat16),
- **exactly one device** (this port is single-GPU / single-CPU),
- a HuggingFace config object (the patched one under `model_assets/`).

`max_seq_len` is taken from `max_len` so the batcher and the graph agree.

In [0]:
#| echo: false
#| output: asis
show_doc(SmiTedModelConfig)

---

### SmiTedModelConfig

```python
def SmiTedModelConfig(
    dtype:DType, device:DeviceRef, huggingface_config:SmiTedHFConfig, max_seq_len:int
)->None:
```

*MAX architecture config: dtype, device, HF hyperparams, bounded seq len.*

### Checkpoint

In one sentence: *config turns Hub JSON into dtype + device + `max_len`, so every later module speaks the same sizes.*